# Slovak AI Olympiad — Task 3 (Part 2): Zero-Shot Diacritics Restoration via SlovakBERT
**Author:** Kristian Hudák  
**Competition:** National Olympiad in Artificial Intelligence (OAI) — Home Qualifier Round

This notebook implements a zero-shot inference pipeline for restoring missing Slovak diacritics:
1. Loading the pre-trained `gerulata/slovakbert` Masked Language Model without parameter updates
2. Word-level iterative masking: masking one word at a time with `[MASK]` and predicting context-aware candidates
3. Candidate matching: filtering `top_k=50` predictions to find the highest probability word whose stripped form matches the input
4. Character-level accuracy evaluation on Slovak diacritic characters (`a, c, d, e, i, l, n, o, r, s, t, u, y, z`)
5. Reaching ~91.50% zero-shot character accuracy and exporting predictions to TSV


In [1]:
# 1. Load pretrained SlovakBERT Masked LM and tokenizer
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import torch.nn.functional as F
import pandas as pd
import unicodedata

# Select compute device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("gerulata/slovakbert")
model = AutoModelForMaskedLM.from_pretrained("gerulata/slovakbert")

model.to(device)
model.eval() # Model prepnutý do režimu evaluácie
print(f"Model SlovakBERT je pripravený na zariadení: {device}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Model SlovakBERT je pripravený na zariadení: cuda


In [2]:
# 2. Zero-shot diacritics restoration heuristics and character accuracy metrics
def odstran_diakritiku(text):
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

def oprav_vetu_zeroshot_manualne(veta_bez):
    slova = veta_bez.split()
    veta_v_procese = slova.copy()

    for i in range(len(slova)):
        povodny_vstup = slova[i].lower()
        testovacia = veta_v_procese.copy()
        testovacia[i] = tokenizer.mask_token
        maskovana_veta = " ".join(testovacia)

        inputs = tokenizer(maskovana_veta, return_tensors="pt").to(device)
        mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

        mask_token_logits = logits[0, mask_token_index, :]
        top_50_results = torch.topk(mask_token_logits, 50, dim=1)
        top_50_ids = top_50_results.indices[0].tolist()

        najlepsi_kandidat = veta_v_procese[i]

        for token_id in top_50_ids:
            kandidat = tokenizer.decode([token_id]).strip()
            # Logika: Ak sa kandidát po odstránení diakritiky zhoduje so vstupom, vyberáme ho
            if odstran_diakritiku(kandidat).lower() == povodny_vstup:
                if kandidat.lower() != povodny_vstup:
                    najlepsi_kandidat = kandidat
                    break
        veta_v_procese[i] = najlepsi_kandidat

    return " ".join(veta_v_procese)

def vypocitaj_accuracy_vety(ref_veta, pred_veta):
    mozna_diakritika = set("acdeilnorstuyz")
    spravne = 0
    celkovo_pozicii = 0

    for r, p in zip(ref_veta.lower(), pred_veta.lower()):
        zakladny_znak = odstran_diakritiku(r)
        if zakladny_znak in mozna_diakritika:
            celkovo_pozicii += 1
            if r == p:
                spravne += 1
    return spravne, celkovo_pozicii


In [4]:
# 3. Run zero-shot restoration test on sample sentences
df_path = '/content/vety_dataset (1).xlsx'
df = pd.read_excel(df_path)

ukazky = df.sample(5)
print("Spúšťam manuálnu Zero-shot opravu (bez pipeline)... \n")

for _, row in ukazky.iterrows():
    vstup = row['veta_bez_diakritiky']
    skutocnost = row['veta_s_diakritikou']
    predikcia = oprav_vetu_zeroshot_manualne(vstup)

    print(f"VSTUP:  {vstup}")
    print(f"BERT:   {predikcia}")
    print(f"REÁLNE: {skutocnost}")
    print("-" * 80)


Spúšťam manuálnu Zero-shot opravu (bez pipeline)... 

VSTUP:  Vyssie spomenute rozdiely povodnych nareci boli v 1. pol. 2. tis. pred.
BERT:   Vyššie spomenuté rozdiely povodnych nareci boli v 1. pol. 2. tis. pred.
REÁLNE: Vyššie spomenuté rozdiely pôvodných nárečí boli v 1. pol. 2. tis. pred.
--------------------------------------------------------------------------------
VSTUP:  Pri bojoch v juznej casti ZSSR nemecke vojska nedobyli Kaukaz a navyse sa zaplietli do dlhotrvajucich a vycerpavajucich bojov o Stalingrad, ktore sa pokusali obaja diktatori z prestiznych i strategickych dovodov udrzat za kazdu cenu.
BERT:   Při bojoch v južnej časti ZSSR nemecké vojská nedobyli Kaukaz a navyše sa zaplietli do dlhotrvajucich a vycerpavajucich bojov o Stalingrad, ktoré sa pokúšali obaja diktatori z prestiznych i strategických dôvodov udržať ža každú cenu.
REÁLNE: Pri bojoch v južnej časti ZSSR nemecké vojská nedobyli Kaukaz a navyše sa zaplietli do dlhotrvajúcich a vyčerpávajúcich bojov o Stali

In [5]:
# 4. Process full validation split (10% of dataset) with progress logging
from tqdm import tqdm
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
val_df = val_df.reset_index(drop=True)

tsv_data = []
celkove_spravne = 0
celkove_pozicie = 0
mozna_diakritika = set("acdeilnorstuyz")

print(f"Spúšťam spracovanie {len(val_df)} viet...")

for i, row in tqdm(val_df.iterrows(), total=len(val_df)):
    vstup = row['veta_bez_diakritiky']
    realita = row['veta_s_diakritikou']
    predikcia = oprav_vetu_zeroshot_manualne(vstup)

    tsv_data.append({"id": i + 1, "text": predikcia})

    # Priebežný výpočet globálnej accuracy
    for r, p in zip(realita.lower(), predikcia.lower()):
        zakladny_znak = odstran_diakritiku(r)
        if zakladny_znak in mozna_diakritika:
            celkove_pozicie += 1
            if r == p:
                celkove_spravne += 1


Spúšťam spracovanie 2000 viet...


100%|██████████| 2000/2000 [06:54<00:00,  4.83it/s]


In [6]:
# 5. Compute overall character accuracy and export predictions to TSV
from google.colab import files

predictions_df = pd.DataFrame(tsv_data)
predictions_df.to_csv('predictions.tsv', sep='\t', index=False)

finalna_accuracy = (celkove_spravne / celkove_pozicie) * 100 if celkove_pozicie > 0 else 0

print("\n" + "="*50)
print(f"HOTOVO. Súbor 'predictions.tsv' bol vytvorený.")
print(f"KONEČNÁ ZERO-SHOT ACCURACY: {finalna_accuracy:.2f} %")
print("="*50)

# Stiahnutie súboru do lokálneho PC
files.download('predictions.tsv')



HOTOVO. Súbor 'predictions.tsv' bol vytvorený.
KONEČNÁ ZERO-SHOT ACCURACY: 92.98 %


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>